# Notebook 02 — Baseline (DummyRegressor)

## Objectif
Construire un modèle de référence minimal qui prédit la moyenne de `target` pour toutes les lignes. Ce score sert de plancher : tout modèle utile doit le battre.

## Fichiers utilisés
- `data/X_train.csv` — variables explicatives
- `data/y_train.csv` — variable cible
- `data/X_test.csv` — visites à prédire

## Résultat attendu
- RMSE de la baseline
- Rapport Skore Hub (clé `01_dummy`)
- Fichier `submissions/01_dummy.csv`

## Notebook suivant
`03_ridge.ipynb` — régression Ridge avec imputation.

## 1. Imports

In [1]:
import json
import os
from pathlib import Path

import pandas as pd
import skore
from sklearn.dummy import DummyRegressor

## 2. Connexion à Skore Hub

In [2]:
config = json.loads(Path("../.skore").read_text())
os.environ["SKORE_HUB_API_KEY"] = config["api_key"]
skore.login()
hub_project = skore.Project(
    "bobathon-esilv",
    mode="hub",
    workspace=config["workspace"],
)
print(f"Connecté au Hub : workspace={hub_project.workspace}")

╭───────────────────────────────── Login to Skore Hub ─────────────────────────────────╮
│                                                                                      │
│                        Successfully logged in, using API key.                        │
│                                                                                      │
╰──────────────────────────────────────────────────────────────────────────────────────╯

Connecté au Hub : workspace=BERNIE MADOFF


## 3. Chargement des données

In [3]:
DATA_DIR = Path("../data")

X_train = pd.read_csv(DATA_DIR / "X_train.csv")
y_train = pd.read_csv(DATA_DIR / "y_train.csv")
X_test = pd.read_csv(DATA_DIR / "X_test.csv")

print(f"X_train : {X_train.shape}")
print(f"y_train : {y_train.shape}")
print(f"X_test  : {X_test.shape}")

X_train : (44590, 13)
y_train : (44590, 2)
X_test  : (11013, 13)


## 4. Fusion et vérifications

In [4]:
visits = X_train.merge(
    y_train,
    on="Index",
    how="inner",
    validate="one_to_one",
)

assert len(visits) == len(X_train)
assert len(visits) == len(y_train)
assert visits["target"].notna().all()

print(f"visits : {visits.shape} — fusion OK")

visits : (44590, 14) — fusion OK


## 5. Sélection des variables et préparation de X, y

In [5]:
feature_cols = [
    "sexM",
    "age_at_diagnosis",
    "age",
    "ledd",
    "time_since_intake_on",
    "time_since_intake_off",
    "on",
    "off",
]

X = visits[feature_cols]
y = visits["target"]

print(f"X : {X.shape}, y : {y.shape}")

X : (44590, 8), y : (44590,)


## 6. Modèle baseline — DummyRegressor

Le `DummyRegressor(strategy="mean")` prédit systématiquement la moyenne de `target` observée dans l'ensemble d'entraînement, indépendamment de toutes les variables explicatives. Il sert de plancher absolu : tout modèle réel doit produire une RMSE inférieure.

In [6]:
dummy = DummyRegressor(strategy="mean")
print(dummy)

DummyRegressor()


## 7. Évaluation avec Skore

In [7]:
report = skore.evaluate(
    dummy,
    X,
    y,
)

rmse_baseline = report.metrics.rmse()
print(f"RMSE baseline (DummyRegressor) : {rmse_baseline:.4f}")

RMSE baseline (DummyRegressor) : 16.4779


## 8. Rapport Skore Hub

In [8]:
hub_project.put("01_dummy", report)
print("Rapport envoyé à Skore Hub sous la clé : 01_dummy")

Output()

Consult your report at https://skore.probabl.ai/BERNIE 
MADOFF/bobathon-esilv/estimators/51074

Rapport envoyé à Skore Hub sous la clé : 01_dummy


## 9. Entraînement sur toutes les données et génération de la soumission

In [9]:
dummy.fit(X, y)

X_test_model = X_test[feature_cols].copy()
predictions = dummy.predict(X_test_model)

submission = X_test[["Index"]].copy()
submission["target"] = predictions

print(f"Prédictions : {len(predictions)} lignes")
print(f"Valeur prédite (constante) : {predictions[0]:.4f}")

Prédictions : 11013 lignes
Valeur prédite (constante) : 37.4715


## 10. Vérifications de la soumission

In [10]:
assert list(submission.columns) == ["Index", "target"], "Colonnes incorrectes"
assert len(submission) == len(X_test), "Nombre de lignes incorrect"
assert submission["Index"].equals(X_test["Index"]), "Index ne correspondent pas"
assert submission["target"].notna().all(), "Valeurs manquantes dans target"

print("Vérifications soumission : OK")
display(submission.head())

Vérifications soumission : OK


,Index,target
0,66,37.471527
1,67,37.471527
2,68,37.471527
3,69,37.471527
4,70,37.471527


## 11. Enregistrement du fichier de soumission

In [11]:
submissions_dir = Path("../submissions")
submissions_dir.mkdir(exist_ok=True)

submission.to_csv(submissions_dir / "01_dummy.csv", index=False)
print(f"Fichier enregistré : {submissions_dir / '01_dummy.csv'}")

Fichier enregistré : ..\submissions\01_dummy.csv


## 12. Conclusion

La baseline `DummyRegressor(strategy="mean")` prédit la moyenne de `target` sur tout le jeu de test.

**Points clés :**
- La RMSE baseline correspond à l'écart-type de `target` (par définition d'un prédicteur par la moyenne).
- Tout modèle réel (Ridge, HGBR…) doit produire une RMSE strictement inférieure.
- Le rapport a été envoyé sur Skore Hub sous la clé `01_dummy`.
- La soumission générée est conforme au format Kaggle.

**Prochaine étape** : `03_ridge.ipynb` — pipeline `SimpleImputer + Ridge` avec plusieurs valeurs de `alpha`.